# NumPy Fundamentals — Basic Concepts

These notes introduce the foundations of NumPy used in later numerical and quantum-computing work.

**Topics:** ndarray, dtype, memory layout and strides, views and copies.

Run the code cells from top to bottom. All examples use NumPy only.

## Part 1 — ndarray: NumPy's core data container

An ndarray (N-dimensional array) is NumPy's fundamental data container. Ordinary numerical arrays usually store elements of a common dtype in a regular data buffer, enabling efficient vectorized operations.

| Python list | NumPy ndarray |
|---|---|
| Can contain different object types | Usually has one dtype |
| Stores references to Python objects | Numeric elements use a regular data buffer |
| Nested lists need not be rectangular | A regular array has a well-defined shape |
| Element-wise work often uses Python loops | Many operations run in optimized native code |

NumPy also supports object and structured dtypes, but these do not have the same compact layout as ordinary numeric arrays.

### 1.1 Inspecting an array

In [ ]:
import numpy as np

a = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.int64)

print(type(a))       # numpy.ndarray
print("shape:", a.shape)
print("ndim:", a.ndim)
print("dtype:", a.dtype)
print("size:", a.size)
print("itemsize:", a.itemsize)
print("nbytes:", a.nbytes)

- shape: a tuple giving the length of each axis; (2, 3) means 2 rows and 3 columns.
- ndim: number of axes.
- size: total number of elements; here, 2 × 3 = 6.
- dtype: type used to interpret each element.
- itemsize: bytes per element.
- nbytes: bytes in the element data buffer; for this numeric array, size * itemsize.

shape describes each axis, ndim counts axes, and size counts all elements.

### 1.2 Scalars, vectors, matrices, and identity matrices

In [ ]:
scalar = np.array(7)
vector = np.array([7, 8, 9])
row = np.array([[7, 8, 9]])
column = np.array([[7], [8], [9]])
matrix = np.array([[1, 2, 3], [4, 5, 6]])
identity = np.eye(3)

for name, x in [("scalar", scalar), ("vector", vector), ("row", row),
                ("column", column), ("matrix", matrix), ("identity", identity)]:
    print(f"{name}: shape={x.shape}, ndim={x.ndim}, dtype={x.dtype}")
    print(x, "\n")

**Important:** a one-dimensional array with shape (3,) is neither a row matrix (1, 3) nor a column matrix (3, 1). This distinction matters in linear algebra and matrix multiplication.

### 1.3 Indexing: NumPy scalars and subarrays

In [ ]:
a = np.array([[1, 2], [3, 4]], dtype=np.int64)

element = a[0, 0]  # NumPy scalar
row = a[0]         # one-dimensional ndarray

print(element, type(element))
print(row, type(row), row.shape)

Integer indexing that selects one element typically returns a NumPy scalar. Selecting a row or slice typically returns an ndarray. Check type(x) and x.shape when unsure.

### 1.4 nbytes is not the total memory used by an array object

a.nbytes reports the size of the element data buffer. It excludes the Python array object, its dtype object, and allocator overhead.

For dtype=object, nbytes mainly counts the object references stored in the array buffer; it does not include the memory used by the referenced Python objects.

## Part 2 — dtype: how NumPy interprets elements

dtype means data type object. It describes how the bytes representing each element should be interpreted, including the kind of value, its size, and (where relevant) byte order. The same bytes can represent different values under different dtypes.

### 2.1 Common dtype families

| Type family | Typical use |
|---|---|
| bool_ | True / False |
| Signed integers | Positive and negative whole numbers |
| Unsigned integers | Non-negative integers; often pixels or bit-level data |
| Floating-point | Approximate real numbers |
| Complex | Real and imaginary components |
| String | Fixed-width byte or Unicode strings |
| object | Python object references; flexible, but usually slower for numerical work |
| datetime64 / timedelta64 | Dates, times, and time intervals |

### 2.2 Element sizes

In [ ]:
types = [np.int8, np.int16, np.int32, np.int64,
         np.float16, np.float32, np.float64,
         np.complex64, np.complex128]

for dtype in types:
    x = np.array([0], dtype=dtype)
    print(f"{str(x.dtype):<12} {x.itemsize} byte(s)")

Typical sizes: int8 = 1 byte, int16 = 2, int32 and float32 = 4, int64 and float64 = 8, complex64 = 8, and complex128 = 16. Inspect itemsize directly rather than relying on memory.

### 2.3 Integers: fixed-width ranges

NumPy integer dtypes have fixed widths. For example, int8 is an 8-bit signed integer with range −128 to 127. Python's built-in int supports arbitrary precision (subject to available memory).

Choose a dtype based on value range, precision, memory, and later operations. uint8 is common for image pixels; int32 and int64 are common integer types. For many physical calculations, float64 is a common default. Smaller is not automatically better.

### 2.4 Floating-point numbers: finite precision

In [ ]:
x32 = np.array([0, 1], dtype=np.float32)
x64 = np.array([0, 1], dtype=np.float64)

print(x32, x32.dtype, x32.itemsize)
print(x64, x64.dtype, x64.itemsize)
print("float32 epsilon:", np.finfo(np.float32).eps)
print("float64 epsilon:", np.finfo(np.float64).eps)

float32 uses 32 bits and typically provides about 6–7 decimal significant digits; float64 uses 64 bits and typically provides about 15–16. Floating-point values are approximations, so small rounding differences are normal. np.finfo(dtype).eps gives the spacing near 1.

### 2.5 Complex numbers

A complex number has real and imaginary parts. Python writes the imaginary unit as 1j (mathematically, i). complex64 uses two 32-bit floating-point components; complex128 uses two 64-bit components.

Complex arrays are useful for representing quantum-state amplitudes and quantum gates.

In [ ]:
# A normalized one-qubit state represented by a complex array
psi = np.array([1 / np.sqrt(2), 1j / np.sqrt(2)], dtype=np.complex128)

print(psi)
print("real:", psi.real)
print("imag:", psi.imag)
print("dtype:", psi.dtype, "itemsize:", psi.itemsize)

inner_product = np.vdot(psi, psi)  # conjugates the first argument
print("<psi|psi>:", inner_product)
print("normalized:", np.isclose(inner_product, 1.0))

### 2.6 Dtype inference and explicit specification

In [ ]:
a = np.array([1, 2, 3])
b = np.array([1.0, 2.0, 3.0])
c = np.array([1, 2, 3], dtype=np.float32)
d = np.array([1, 2, 3], dtype=np.complex128)

for x in (a, b, c, d):
    print(x, x.dtype)

If dtype is omitted, NumPy infers a type from the input values. Inspect the inferred dtype while exploring; specify it explicitly when precision, memory, or compatibility matters.

### 2.7 astype: converting an array

In [ ]:
x = np.array([1.8, 2.2, -3.7], dtype=np.float64)
y = x.astype(np.int32)

print(x, x.dtype)
print(y, y.dtype)
print(np.can_cast(np.int32, np.float64, casting="safe"))
print(np.can_cast(np.float64, np.int32, casting="safe"))

Converting floats to integers typically truncates toward zero: 1.8 becomes 1, and -3.7 becomes -3. By default, astype returns a new array.

Do not confuse convertible with lossless. Conversions can discard fractional parts or fail to represent values outside the target range. Consider precision, range, and conversion rules before casting.

## Part 3 — Memory layout: shape and strides

Memory is addressed linearly. NumPy uses metadata such as shape and strides to map multidimensional indices to byte offsets.

For index (i₀, i₁, …, iₙ₋₁), the offset from the array's data pointer is

offset = i₀s₀ + i₁s₁ + … + iₙ₋₁sₙ₋₁,

where each stride sₖ is measured in bytes.

- shape specifies the valid index range on each axis.
- strides specifies the byte step when an index advances by one.
- dtype.itemsize gives the size of one element for ordinary fixed-width numeric dtypes.

### 3.1 C order and Fortran (F) order

For a two-dimensional array, C order (row-major) stores elements in each row next to one another; Fortran order (column-major) stores elements in each column next to one another. np.asfortranarray creates a Fortran-contiguous array when needed.

In [ ]:
a = np.array([[10, 20, 30], [40, 50, 60]], dtype=np.int64)
f = np.asfortranarray(a)

print("C strides:", a.strides)
print("F strides:", f.strides)
print("C flags:", a.flags.c_contiguous, a.flags.f_contiguous)
print("F flags:", f.flags.c_contiguous, f.flags.f_contiguous)

### 3.2 Why slicing and transposition change strides

In [ ]:
a = np.arange(12, dtype=np.int64).reshape(3, 4)

print("a:", a.shape, a.strides)
print("a.T:", a.T.shape, a.T.strides)
print("a[:, ::2]:", a[:, ::2].shape, a[:, ::2].strides)
print("a[::-1]:", a[::-1].shape, a[::-1].strides)

Transposition usually changes axis metadata and corresponding strides without moving the data. a[:, ::2] selects every other column, doubling the stride along that axis. a[::-1] reverses axis 0 and typically produces a negative stride: increasing the index moves toward a lower memory address.

### 3.3 flags: memory properties

In [ ]:
a = np.arange(12, dtype=np.float64).reshape(3, 4)
t = a.T

print(a.flags.c_contiguous, a.flags.f_contiguous)
print(t.flags.c_contiguous, t.flags.f_contiguous)
print(a.flags.owndata)
print(a.flags.writeable)

| Flag | Meaning |
|---|---|
| c_contiguous | Contiguous in C order |
| f_contiguous | Contiguous in Fortran order |
| owndata | The array owns its data buffer |
| writeable | The array's data can be modified |

## Part 4 — Views and copies

A view is a new array object with its own metadata (such as shape and strides) that shares the underlying data buffer with another array. A copy has independent element storage; modifying it does not modify the original array.

### 4.1 Basic slicing often returns a view

In [ ]:
a = np.array([10, 20, 30, 40, 50])
v = a[1:4]

print(v)
print(np.shares_memory(a, v))
v[0] = 999
print("a:", a)
print("v:", v)

Here v and a share data, so assigning to v[0] also changes the corresponding element in a.

### 4.2 copy() creates independent storage

In [ ]:
a = np.array([10, 20, 30, 40, 50])
c = a[1:4].copy()

print(np.shares_memory(a, c))  # False
c[0] = 999
print("a:", a)
print("c:", c)

The copy uses independent element storage. The trade-off is the allocation of memory and copying of data.

### 4.3 Advanced indexing typically returns a copy

In [ ]:
a = np.arange(10)

basic = a[2:7]          # Basic slicing: usually a view
fancy = a[[2, 4, 6]]    # Integer-array indexing: copy
mask = a[a % 2 == 0]    # Boolean indexing: copy

print(np.shares_memory(a, basic))
print(np.shares_memory(a, fancy))
print(np.shares_memory(a, mask))

Rule of thumb: basic slicing usually returns a view; integer-array and Boolean indexing typically return copies. When uncertain, use np.shares_memory(x, y).

### 4.4 Memory behavior of common operations

| Operation | Typical behavior |
|---|---|
| a.T / a.transpose() | Usually a view |
| a.reshape() | View or copy, depending on layout |
| a.ravel() | View if possible; otherwise copy |
| a.flatten() | Copy |
| a.astype(dtype) | Copy by default |
| a.copy() | Copy |

Do not assume reshape always returns a view. If correctness depends on memory sharing, check with np.shares_memory or explicitly copy when independent storage is required.

## Practical checklist

When an array behaves unexpectedly, inspect:

1. x.shape, x.ndim, x.size — logical dimensions.
2. x.dtype, x.itemsize, x.nbytes — data type and buffer size.
3. x.strides, x.flags — memory layout and contiguity.
4. np.shares_memory(x, y) — whether two arrays share data.

If later modifications must not affect the original, use y = x.copy().

## Appendix — Quick reference

| Property / method | Meaning |
|---|---|
| x.shape | Length of each axis |
| x.ndim | Number of axes |
| x.size | Total number of elements |
| x.dtype | Element data type |
| x.itemsize | Bytes per element |
| x.nbytes | Bytes in the element data buffer |
| x.strides | Byte step along each axis |
| x.flags | Memory-layout and writeability flags |
| np.shares_memory(x, y) | Whether arrays share memory |
| x.copy() | Create independent element storage |
| x.astype(dtype) | Convert to another dtype |
| x.reshape(shape) | Change shape; may view or copy |
| x.ravel() / x.flatten() | Flatten; ravel may view or copy, flatten copies |

### Official references

- https://numpy.org/doc/stable/reference/arrays.ndarray.html
- https://numpy.org/doc/stable/reference/arrays.dtypes.html
- https://numpy.org/doc/stable/user/basics.copies.html
- https://numpy.org/doc/stable/user/basics.types.html

**Scope:** these notes cover ndarray, dtype, memory layout, strides, views, and copies. Broadcasting, universal functions, aggregation, linear algebra, random numbers, FFT, and I/O will be covered in later modules.